# Length distribution scaling debugging

It appears that there may be a bug with the length distribution scaling. Following scaling, segments of the genome do not normalize to the expected shape when compared to the replication profile deconvolution.

It's possible that segments of the genome are disproportionately influence the total length distribution curves. Check chrXII for example for omission.


In [1]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [14]:
from src.timer import Timer

def compute_len_dist_for_replicate(replicate, timepoints=None, chroms=None):

    timer = Timer()

    from src.chromatin_metrics import ChromatinMetrics, fragment_lengths_definitions
    small_len, mid_len, nuc_len = fragment_lengths_definitions()
    chrom_metrics = ChromatinMetrics(small_len, mid_len, nuc_len)

    chrom_metrics.set_chrom(chrom=1, replicate=replicate)
    
    if timepoints is None:
        timepoints = chrom_metrics.chrom_reads['sample'].unique()
    timepoint_counts = {}
    
    print(f"Computing the length distribution for replicate {replicate}")

    timepoint_chrom_counts = {}
    if chroms is None:
        chroms = np.arange(1, 17)
    
    for timepoint in timepoints:
        cumulative_counts = None
        
        print(f"   t={timepoint}. For Chr: ", end=" ")
        
        chrom_counts = {}
        for chrom in chroms:

            print(f"{chrom}", end=", ")
            chrom_metrics.set_chrom(chrom=chrom, replicate=replicate)
            lens, counts = chrom_metrics.compute_length_hist(timepoint)
            chrom_counts[chrom] = counts
            
            if cumulative_counts is None:
                cumulative_counts = counts
            else:
                cumulative_counts = cumulative_counts + counts

        timepoint_chrom_counts[timepoint] = chrom_counts
        timepoint_counts[timepoint] = cumulative_counts
        timepoints_counts_df = pd.DataFrame(timepoint_counts)
        print()

    print(f"Completed {timer.get_time()}")
    
    # Create a data frame of the counts per timepoint and chromosome
    total_df = pd.DataFrame()
    for timepoint in timepoints:
        timepoint_len_counts_df = pd.DataFrame(timepoint_chrom_counts[timepoint]).T
        timepoint_len_counts_df.index.name = 'chrom'
        timepoint_len_counts_df = timepoint_len_counts_df.reset_index()
        timepoint_len_counts_df['timepoint'] = timepoint
        timepoint_len_counts_df = timepoint_len_counts_df.set_index(['timepoint', 'chrom'])
        total_df = pd.concat([total_df, timepoint_len_counts_df])

    return total_df

In [172]:
from src.global_config import GlobalConstants

timepoints = GlobalConstants.CHROM_WT1_TIMEPOINTS
chroms = range(1, 17)

rep1_total_df = \
    compute_len_dist_for_replicate(1, timepoints=timepoints, chroms=chroms)
rep1_total_df.to_csv('datasets/computed_mnase/rep1_length_counts.csv')


Computing the length distribution for replicate 1
   t=0. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=10. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=20. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=30. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=40. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=50. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=60. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=70. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=80. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=90. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=100. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=110. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=120. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10

In [268]:
timepoints2 = GlobalConstants.CHROM_WT2_TIMEPOINTS
rep2_total_df = \
    compute_len_dist_for_replicate(2, timepoints=timepoints2, chroms=chroms)

Computing the length distribution for replicate 2
   t=0. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=10. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=20. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=30. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=40. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=50. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=60. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=70. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=80. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=90. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=100. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=110. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 
   t=120. For Chr:  1, 2, 3, 4, 5, 6, 7, 8, 9, 10

In [267]:
rep2_total_df.to_csv('datasets/computed_mnase/rep2_length_counts.csv')

In [269]:
# Create a target distribution by merging the rep1 and rep2 distributions.